In [ ]:
from gen_prompts import NORMAL_PROMPT,RESPONSE_FORMAT,TOPICS,ATTACK_PROMPT,ATTACK_TYPES
from dotenv import load_dotenv
import json, re
import anthropic
import random
load_dotenv()
normal_dataset_path = 'datasets/normal_dataset.json'
guardrail_dataset_path = 'datasets/guardrail_dataset.json'
guardrail2_dataset_path = 'datasets/guardrail_dataset2.json'
random.seed()

## Functions to clean and store datasets

In [ ]:
def format_response(dataset):
    cleaned_list = []
    for i,resp in enumerate(dataset):
        cleaned = re.sub(r"^```.*|```$", "", resp).strip()
        try:
            cleaned_list += json.loads(cleaned)
        except:
            print(f"Could not perform json loads[{i}]")
            print("RESPONSE:", resp)
    
    print("LEN: ",len(cleaned_list))
    return cleaned_list


def update_dataset(new_dataset,path):
    dataset = []
    with open(path,'r') as file:
        dataset = json.load(file) + new_dataset

    with open(path,'w',encoding="utf-8") as file:
        json.dump(dataset,file,ensure_ascii=False)

    print("Number of rows in dataset: ",len(dataset))

## Normal conversations

In [ ]:
normal_resp = []

for i,topic in enumerate(TOPICS):
    message = NORMAL_PROMPT.format(random.randint(1,5),topic,RESPONSE_FORMAT)

    response = anthropic.Anthropic().messages.create(
        model="claude-sonnet-4-5",
        max_tokens=2048,
        messages=[{"role": "user", "content": message}
        ]   
    )
    normal_resp.append(response.content[0].text)
    

In [ ]:
normal_cleaned = format_response(normal_resp)
update_dataset(normal_cleaned,normal_dataset_path)

## Guardrail2 conversations

In [ ]:
guardrail2_resp = []

for i,attack in enumerate(ATTACK_TYPES):
    message = ATTACK_PROMPT.format(random.randint(1,5),attack,RESPONSE_FORMAT)

    response = anthropic.Anthropic().messages.create(
        model="claude-sonnet-4-5",
        max_tokens=2048,
        messages=[{"role": "user", "content": message}
        ]   
    )
    guardrail2_resp.append(response.content[0].text)


In [ ]:
guardrail2_cleaned = format_response(guardrail2_resp)
update_dataset(guardrail2_cleaned,guardrail2_dataset_path)

## Upload Dataset to HuggingFace

In [ ]:
# Login to HuggingFace
from huggingface_hub import login
import os

login(token=os.environ['HF_TOKEN'])

In [ ]:
from datasets import Dataset, DatasetDict
import json

# Load your datasets
with open(guardrail_dataset_path, 'r') as f:
    guardrail_data = json.load(f)

with open(guardrail2_dataset_path, 'r') as f:
    guardrail_data2 = json.load(f)

with open(normal_dataset_path, 'r') as f:
    normal_data = json.load(f)

# Create HuggingFace datasets
guardrail_dataset = Dataset.from_list(guardrail_data)
guardrail_dataset2 = Dataset.from_list(guardrail_data2)
normal_dataset = Dataset.from_list(normal_data)

# Combine into a DatasetDict (optional - allows multiple splits)
dataset_dict = DatasetDict({
    "guardrail_v1": guardrail_dataset,
    "guardrail_v2": guardrail_dataset2,
    "normal_conversations": normal_dataset
})

print(f"Guardrail v1: {len(guardrail_dataset)} examples")
print(f"Guardrail v2: {len(guardrail_dataset2)} examples")
print(f"Normal conversations: {len(normal_dataset)} examples")

In [ ]:
dataset_dict.push_to_hub(
    repo_id="alindstroem89/guardrail-training-dataset",
    private=False
)

### Loading the Dataset from HuggingFace

In [ ]:
from datasets import load_dataset

# Load the entire dataset
dataset = load_dataset("your-username/guardrail-training-dataset")

# Access individual splits
guardrail_v1 = dataset["guardrail_v1"]
guardrail_v2 = dataset["guardrail_v2"]
normal_conv = dataset["normal_conversations"]